# 04 — Interactive Generation

## Scientific question

**What does one frozen V1 sampling trajectory look like in practice?**

Live sampling is deliberately disabled in the release configuration. The notebook remains executable and documents the exact opt-in backend contract without fabricating an output.

## Environment & reproducibility setup

In [1]:
from __future__ import annotations

import json
import platform
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import torch


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "tiny_avatar_diffusion").is_dir():
            return candidate
    raise RuntimeError(
        "Repository root not found. Run this notebook from inside the Tiny Avatar Diffusion repository."
    )


PROJECT_ROOT = find_project_root()
environment = {
    "python": platform.python_version(),
    "package": version("tiny-avatar-diffusion"),
    "project_root": PROJECT_ROOT.name,
    "project_markers_verified": True,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "selected_device": "cpu",
}
environment

{'python': '3.12.4',
 'package': '0.0.0',
 'project_root': 'DL_Project',
 'project_markers_verified': True,
 'torch': '2.5.1+cu121',
 'cuda_available': True,
 'selected_device': 'cpu'}

## Optional live-experiment configuration

In [2]:
RUN_LIVE_GENERATION = False
RUN_PAIRED_COMPARISON = False
NOTEBOOK_SEED = 2026
REQUESTED_DEVICE = "auto"
SNAPSHOT_TIMESTEPS = (999, 749, 499, 249, 0)

print("Live generation disabled by default." if not RUN_LIVE_GENERATION else "Live generation enabled.")
print("Paired comparison disabled by default." if not RUN_PAIRED_COMPARISON else "Paired comparison enabled.")

Live generation disabled by default.
Paired comparison disabled by default.


## Canonical inference contract and prompt construction

The backend owns checkpoint validation, device resolution, prompt validation, DDPM sampling, paired RNG semantics, evaluator preprocessing, snapshots, and provenance. This notebook calls those interfaces; it does not reproduce them.

In [3]:
from tiny_avatar_diffusion.inference import (
    CONDITIONAL_CHECKPOINT,
    CONDITIONAL_SHA256,
    EVALUATOR_CHECKPOINT,
    EVALUATOR_SHA256,
    UNCONDITIONAL_CHECKPOINT,
    UNCONDITIONAL_SHA256,
    PromptAttributes,
)

REQUESTED_ATTRIBUTES = PromptAttributes(
    face_shape=5,
    glasses=6,
    hair_color=9,
    face_color=0,
)
checkpoint_identity = {
    "conditional": {"filename": CONDITIONAL_CHECKPOINT.name, "sha256": CONDITIONAL_SHA256},
    "unconditional": {"filename": UNCONDITIONAL_CHECKPOINT.name, "sha256": UNCONDITIONAL_SHA256},
    "evaluator": {"filename": EVALUATOR_CHECKPOINT.name, "sha256": EVALUATOR_SHA256},
}
{
    "canonical_prompt": REQUESTED_ATTRIBUTES.caption,
    "seed": NOTEBOOK_SEED,
    "device_policy": REQUESTED_DEVICE,
    "snapshot_timesteps": SNAPSHOT_TIMESTEPS,
    "checkpoint_identity": checkpoint_identity,
}

{'canonical_prompt': 'avatar face_shape_wide glasses_thick_rectangular_glasses hair_color_silver face_color_deep_brown',
 'seed': 2026,
 'device_policy': 'auto',
 'snapshot_timesteps': (999, 749, 499, 249, 0),
 'checkpoint_identity': {'conditional': {'filename': 'conditional_main_ema_step20000.pt',
   'sha256': 'a1abad595c53887b621021b67154a04af2505a4b68e6b9229841601250ca14fb'},
  'unconditional': {'filename': 'unconditional_main_ema_step20000.pt',
   'sha256': '70def31c417a3aabe2fedbc355c2b6fae8478d1f878690f9bbf2d7263cf9fb3e'},
  'evaluator': {'filename': 'dr04_evaluator_epoch41.pt',
   'sha256': '414d2e0bb597c53041f6cf695d965231ab1c4f38c9080b68c06f9c09dcf001dc'}}}

## Optional generation

In [4]:
generation = None
generation_diagnostic = None
if RUN_LIVE_GENERATION:
    from tiny_avatar_diffusion.inference import (
        evaluator_diagnostic,
        generate_conditional,
        load_conditional_bundle,
        load_evaluator_bundle,
    )

    conditional_bundle = load_conditional_bundle(REQUESTED_DEVICE)
    generation = generate_conditional(
        conditional_bundle,
        seed=NOTEBOOK_SEED,
        attributes=REQUESTED_ATTRIBUTES,
        snapshot_timesteps=SNAPSHOT_TIMESTEPS,
    )
    evaluator_bundle = load_evaluator_bundle(REQUESTED_DEVICE)
    generation_diagnostic = evaluator_diagnostic(
        evaluator_bundle, generation.image, requested=REQUESTED_ATTRIBUTES
    )
    display(generation.image)
else:
    print("Live generation disabled by default; no checkpoint was loaded and no sample was generated.")

Live generation disabled by default; no checkpoint was loaded and no sample was generated.


## Denoising trajectory

When enabled, the backend returns detached predicted-$x_0$ observations at the frozen schedule. The initial Gaussian $x_T$ is visualized only after display clipping; it is not modified for sampling. Predicted-$x_0$ snapshots are not $x_t$ states.

In [5]:
if generation is not None:
    from tiny_avatar_diffusion.inference import tensor_to_rgb_image

    labels = ["Initial Gaussian x_T"] + [
        f"Predicted x₀ at t={timestep}" for timestep in SNAPSHOT_TIMESTEPS
    ] + ["Final sampled avatar"]
    images = [tensor_to_rgb_image(generation.initial_x_T[0])]
    images.extend(tensor_to_rgb_image(item.predicted_x_start[0]) for item in generation.snapshots)
    images.append(generation.image)
    fig, axes = plt.subplots(1, len(images), figsize=(14, 2.4))
    for axis, image, label in zip(axes, images, labels, strict=True):
        axis.imshow(image)
        axis.set_title(label, fontsize=8)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Denoising trajectory unavailable because live generation is disabled.")

Denoising trajectory unavailable because live generation is disabled.


## Evaluator diagnostic and provenance

In [6]:
if generation is not None:
    print("LIVE SINGLE-SAMPLE DIAGNOSTIC")
    display(generation_diagnostic)
    print("Scores are uncalibrated evaluator scores, not probabilities of correctness.")
    display(generation.provenance.to_dict())
else:
    print("Evaluator diagnostic unavailable because live generation is disabled.")
    print("A live GenerationResult would carry schema-v1 model, diffusion, runtime, software, and observability provenance.")

Evaluator diagnostic unavailable because live generation is disabled.
A live GenerationResult would carry schema-v1 model, diffusion, runtime, software, and observability provenance.


## Optional paired comparison

The paired backend creates one shared initial $x_T$ and equivalent post-$x_T$ reverse-noise generator states. The conditional and unconditional branches remain distinct models. The result is a **QUALITATIVE PAIRED COMPARISON**, not a dataset-level metric.

In [7]:
paired = None
if RUN_PAIRED_COMPARISON:
    from tiny_avatar_diffusion.inference import (
        generate_paired,
        load_conditional_bundle,
        load_unconditional_bundle,
    )

    paired = generate_paired(
        load_conditional_bundle(REQUESTED_DEVICE),
        load_unconditional_bundle(REQUESTED_DEVICE),
        seed=NOTEBOOK_SEED,
        attributes=REQUESTED_ATTRIBUTES,
        snapshot_timesteps=SNAPSHOT_TIMESTEPS,
    )
    print("QUALITATIVE PAIRED COMPARISON")
    print("Same initial x_T and equivalent reverse-noise RNG state.")
    print("Not a dataset-level metric.")
    display(paired.conditional.image, paired.unconditional.image)
else:
    print("Paired comparison disabled by default; no paired sample was generated.")

Paired comparison disabled by default; no paired sample was generated.


## Interpretation caveats

A single trajectory illustrates the frozen sampling interface but cannot estimate dataset-level quality, diversity, or conditioning adherence. Evaluator outputs are uncalibrated single-sample diagnostics. Scientific conclusions belong to the frozen aggregate evidence inspected in Notebook 03.